# Derivatives Pricing Engine

A complete walkthrough of the engine: theory, formulas, and runnable Python examples.  
Covers options (BS, Heston, SVI, Monte Carlo, CRR), calibration, discount curves, rates, and exotic options.

---

## Table of contents

1. [Setup](#1-setup)
2. [Black-Scholes-Merton](#2-black-scholes-merton)
3. [Implied Volatility](#3-implied-volatility)
4. [Greeks](#4-greeks)
5. [CRR Binomial Tree – European & American](#5-crr-binomial-tree)
6. [Monte Carlo Simulation](#6-monte-carlo)
7. [Volatility Surface (non-parametric)](#7-volatility-surface)
8. [Heston Stochastic Volatility](#8-heston-model)
9. [SVI Smile Parametrisation](#9-svi-model)
10. [Calibration to Live Market Data](#10-calibration)
11. [Self-Updating Pricing Engine](#11-pricing-engine)
12. [Discount Curves & Rates](#12-discount-curves)
13. [Forwards and Futures](#13-forwards-and-futures)
14. [Exotic Options](#14-exotic-options)
15. [Volatility Analysis](#15-volatility-analysis)
16. [Structured Products](#16-structured-products)
17. [Academic Charts](#17-academic-charts)
18. [CLI Reference](#18-cli-reference)

---
## 1. Setup

In [ ]:
import sys, os
# Add project root to path so imports work from the notebook
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

from options.charts import ACADEMIC_STYLE
plt.rcParams.update(ACADEMIC_STYLE)

print('Ready.')

---
## 2. Black-Scholes-Merton

The Black-Scholes-Merton (BSM) model prices **European options** analytically.  
It assumes the underlying follows a Geometric Brownian Motion (GBM) under the risk-neutral measure:

$$dS_t = r\,S_t\,dt + \sigma\,S_t\,dW_t$$

### Pricing formulas

$$C = S_0 N(d_1) - K e^{-rT} N(d_2)$$
$$P = K e^{-rT} N(-d_2) - S_0 N(-d_1)$$

where

$$d_1 = \frac{\ln(S/K) + (r + \sigma^2/2)\,T}{\sigma\sqrt{T}}, \qquad d_2 = d_1 - \sigma\sqrt{T}$$

and $N(\cdot)$ is the standard normal CDF.

### Parameters

| Symbol | Meaning |
|--------|---------|
| $S$ | Current spot price |
| $K$ | Strike price |
| $T$ | Time to expiry (years) |
| $r$ | Continuous risk-free rate |
| $\sigma$ | Implied/realised volatility (annualised) |
| $N(\cdot)$ | Standard normal CDF |

In [ ]:
from options.black_scholes import price, greeks, put_call_parity_check

S, K, T, r, sigma = 100, 100, 1.0, 0.05, 0.20

call_price = price(S, K, T, r, sigma, option='call')
put_price  = price(S, K, T, r, sigma, option='put')

print(f'Call price:  ${call_price:.4f}')
print(f'Put  price:  ${put_price:.4f}')

# Put-call parity:  C - P = S - K*exp(-rT)
pcp = put_call_parity_check(S, K, T, r, call_price, put_price)
print(f'\nPut-call parity error: {pcp["error"]:.2e}  (should be ~0)')

In [ ]:
# Price across a range of strikes to see the skew (flat here, since sigma is constant)
strikes = np.linspace(70, 130, 100)
calls   = [price(S, K_i, T, r, sigma, 'call') for K_i in strikes]
puts    = [price(S, K_i, T, r, sigma, 'put')  for K_i in strikes]

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(strikes, calls, label='Call', color='#0072B2', lw=2)
ax.plot(strikes, puts,  label='Put',  color='#D55E00', lw=2, ls='--')
ax.axvline(S, color='gray', ls=':', lw=1, label=f'S = {S}')
ax.set(xlabel='Strike K ($)', ylabel='Option price ($)',
       title=f'BSM Price vs Strike  [S={S}, T={T}Y, r={r:.0%}, σ={sigma:.0%}]')
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda y, _: f'${y:.2f}'))
ax.legend()
plt.tight_layout()
plt.show()

---
## 3. Implied Volatility

Given a **market price** $C^{mkt}$, find $\hat\sigma$ such that:

$$C_{BS}(\hat\sigma) = C^{mkt}$$

There is no closed form — we solve numerically.

### Algorithm: Newton-Raphson + Brent fallback

**Primary (Newton-Raphson):**

$$\hat\sigma_{n+1} = \hat\sigma_n - \frac{C_{BS}(\hat\sigma_n) - C^{mkt}}{\mathcal{V}(\hat\sigma_n)}$$

Seed from Brenner-Subrahmanyam ATM approximation:
$$\hat\sigma_0 \approx \sqrt{\frac{2\pi}{T}} \cdot \frac{C}{S}$$

**Fallback:** Brent's bracketed root-finding — guaranteed convergence even for deep ITM/OTM options where vega $\approx 0$.

Typical round-trip error: $|\hat\sigma - \sigma^{true}| < 10^{-10}$.

In [ ]:
from options.implied_vol import implied_vol, iv_surface

# Round-trip: price -> IV -> should recover sigma
c     = price(S, K, T, r, sigma, 'call')
iv_hat = implied_vol(S, K, T, r, c, 'call')

print(f'True sigma:       {sigma:.6f}')
print(f'Recovered IV:     {iv_hat:.6f}')
print(f'Round-trip error: {abs(iv_hat - sigma):.2e}')

In [ ]:
# IV from a vol smile (higher IV for OTM strikes — the classic 'smile' effect)
smile_vols = {
    80:  0.30,
    90:  0.24,
    100: 0.20,
    110: 0.22,
    120: 0.26,
}

smile_prices = {K_i: price(S, K_i, T, r, v, 'call') for K_i, v in smile_vols.items()}
recovered    = {K_i: implied_vol(S, K_i, T, r, p, 'call') for K_i, p in smile_prices.items()}

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(list(smile_vols.keys()), [v * 100 for v in smile_vols.values()],
        'o-', color='#0072B2', label='Input vol')
ax.plot(list(recovered.keys()),  [v * 100 for v in recovered.values()],
        'x--', color='#D55E00', label='Recovered IV', ms=10, mew=2)
ax.axvline(S, color='gray', ls=':', lw=1)
ax.set(xlabel='Strike', ylabel='Implied Vol (%)', title='IV Round-trip: Input vs Recovered')
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda y, _: f'{y:.0f}%'))
ax.legend()
plt.tight_layout()
plt.show()

---
## 4. Greeks

Greeks measure the sensitivity of the option price to changes in market parameters.

| Greek | Symbol | Formula | Interpretation |
|-------|--------|---------|----------------|
| Delta | $\Delta_C$ | $N(d_1)$ | $\partial C / \partial S$ — $ change per $1 move in spot |
| Delta | $\Delta_P$ | $N(d_1) - 1$ | Same for put (negative) |
| Gamma | $\Gamma$ | $n(d_1) / (S\sigma\sqrt{T})$ | $\partial^2 C / \partial S^2$ — curvature |
| Vega | $\mathcal{V}$ | $S\,n(d_1)\sqrt{T}$ | $\partial C / \partial \sigma$ — per 1 vol unit |
| Theta | $\Theta_C$ | $-\frac{S\,n(d_1)\,\sigma}{2\sqrt{T}} - r K e^{-rT} N(d_2)$ | $\partial C / \partial t$ — time decay per year |
| Rho | $\rho_C$ | $K T e^{-rT} N(d_2)$ | $\partial C / \partial r$ — rate sensitivity |

where $n(\cdot)$ is the standard normal PDF.

In [ ]:
from options.black_scholes import greeks

g = greeks(S=100, K=100, T=1.0, r=0.05, sigma=0.20)

print('Greeks at ATM (S=K=100, T=1Y, σ=20%, r=5%)\n')
print(f'  Delta (call):   {g["delta_call"]:+.4f}   → $+{g["delta_call"]:.2f} per $1 spot move')
print(f'  Delta (put):    {g["delta_put"]:+.4f}   → ${g["delta_put"]:.2f} per $1 spot move')
print(f'  Gamma:           {g["gamma"]:.5f}   → Delta changes by {g["gamma"]:.4f} per $1 spot move')
print(f'  Vega:          {g["vega"]:7.4f}   → ${g["vega"]/100:.4f} per 1% vol move')
print(f'  Theta (call):  {g["theta_call"]/365:7.4f}   → ${g["theta_call"]/365:.4f} per day')
print(f'  Theta (put):   {g["theta_put"]/365:7.4f}   → ${g["theta_put"]/365:.4f} per day')

In [ ]:
# Plot all 4 Greeks vs spot
spots  = np.linspace(60, 140, 200)
deltas = [greeks(s, 100, 1.0, 0.05, 0.20)['delta_call'] for s in spots]
gammas = [greeks(s, 100, 1.0, 0.05, 0.20)['gamma']      for s in spots]
thetas = [greeks(s, 100, 1.0, 0.05, 0.20)['theta_call'] / 365 for s in spots]
vegas  = [greeks(s, 100, 1.0, 0.05, 0.20)['vega'] / 100 for s in spots]

fig, axes = plt.subplots(2, 2, figsize=(12, 7))
fig.suptitle('Call Option Greeks vs Spot  [K=100, T=1Y, σ=20%, r=5%]', fontsize=13)

for ax, vals, lbl, col in zip(
        axes.flat,
        [deltas, gammas, thetas, vegas],
        ['Δ Delta', 'Γ Gamma', 'Θ Theta / day ($)', 'Vega per 1% σ ($)'],
        ['#0072B2', '#009E73', '#D55E00', '#CC79A7']):
    ax.plot(spots, vals, color=col, lw=2)
    ax.axvline(100, color='gray', ls='--', lw=0.9)
    ax.axhline(0,   color='gray', ls='-',  lw=0.5, alpha=0.4)
    ax.set(xlabel='Spot ($)', ylabel=lbl)
    ax.xaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'${x:.0f}'))

plt.tight_layout()
plt.show()

---
## 5. CRR Binomial Tree

The Cox-Ross-Rubinstein (CRR) tree discretises time into $N$ steps and models the stock as a recombining binomial lattice.

### Tree parameters

$$u = e^{\sigma\sqrt{\Delta t}}, \qquad d = \frac{1}{u}, \qquad p = \frac{e^{r\Delta t} - d}{u - d}$$

where $\Delta t = T / N$.

### Backward induction

At each node $(i, j)$ (time step $i$, up-moves $j$):

$$V_{i,j}^{European} = e^{-r\Delta t}\left[p\,V_{i+1,j+1} + (1-p)\,V_{i+1,j}\right]$$

For **American** options we also allow early exercise at each node:

$$V_{i,j}^{American} = \max\!\left(\text{intrinsic}_{i,j},\; e^{-r\Delta t}\left[p\,V_{i+1,j+1} + (1-p)\,V_{i+1,j}\right]\right)$$

Convergence: CRR → BSM at $O(1/N)$. Use $N \geq 200$ for stable prices.

In [ ]:
from options.binomial_tree import binomial_price

S, K, T, r, sigma = 100, 100, 1.0, 0.05, 0.20

eu = binomial_price(S, K, T, r, sigma, option='call', style='european', n_steps=500)
am = binomial_price(S, K, T, r, sigma, option='put',  style='american', n_steps=500)
bs_ref = price(S, K, T, r, sigma, 'call')

print('CRR Binomial (500 steps)')
print(f'  European call:   ${eu["price"]:.4f}   (BS = ${bs_ref:.4f})')
print(f'  American put:    ${am["price"]:.4f}   (early exercise = ${am["early_exercise"]:.4f}')

In [ ]:
# Convergence plot
step_counts = [5, 10, 20, 50, 100, 200, 500]
eu_prices   = [binomial_price(S, K, T, r, sigma, 'call', 'european', n)['price'] for n in step_counts]
am_prices   = [binomial_price(S, K, T, r, sigma, 'put',  'american', n)['price'] for n in step_counts]

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(step_counts, eu_prices, 'o-', color='#0072B2', label='European call')
ax.plot(step_counts, am_prices, 's-', color='#E69F00', label='American put')
ax.axhline(bs_ref, color='k', ls='--', lw=1.2, label=f'BS ref = ${bs_ref:.4f}')
ax.set(xlabel='Steps N', ylabel='Price ($)', title='CRR Convergence to Black-Scholes')
ax.legend()
plt.tight_layout()
plt.show()

---
## 6. Monte Carlo

Prices options by simulating many risk-neutral GBM paths.

### GBM discretisation (Euler-Maruyama)

$$S_{t+\Delta t} = S_t \exp\!\left[\left(r - \tfrac{1}{2}\sigma^2\right)\Delta t + \sigma\sqrt{\Delta t}\,Z\right], \quad Z \sim \mathcal{N}(0,1)$$

### Variance reduction

**Antithetic variates:** for each $Z$ also simulate $-Z$; average both payoffs.  
**Control variate:** use the known BS European price to reduce MC error by 40-70%.

### Payoffs supported

| `option_type` | Payoff |
|--------------|--------|
| `european_call/put` | $\max(S_T - K, 0)$ |
| `asian_call/put` | $\max(\bar S - K, 0)$ where $\bar S$ = arithmetic mean |
| `barrier_call/put` | European payoff, knocked out if $S_t \leq B$ |
| `lookback_call/put` | $\max(S_T - S_{\min}, 0)$ floating-strike |
| `digital_call/put` | $\mathbf{1}[S_T > K]$ cash-or-nothing |

In [ ]:
from options.monte_carlo import mc_price

S, K, T, r, sigma = 100, 100, 1.0, 0.05, 0.20

results = {}
for ptype in ['european_call', 'asian_call', 'barrier_call', 'digital_call']:
    kw = {'barrier': 80} if 'barrier' in ptype else {}
    res = mc_price(S, K, T, r, sigma, ptype, n_sims=100_000, seed=42, **kw)
    results[ptype] = res
    print(f"{ptype:<20}  ${res['price']:.4f}  ±{res['std_error']:.4f}")

print(f"\nBS reference (European call): ${price(S, K, T, r, sigma, 'call'):.4f}")

In [ ]:
# Visualise GBM paths
n_paths, n_steps = 50, int(T * 252)
dt = T / n_steps
rng = np.random.default_rng(42)
Z   = rng.standard_normal((n_paths, n_steps))
log_ret = (r - 0.5 * sigma**2) * dt + sigma * np.sqrt(dt) * Z
paths   = S * np.exp(np.hstack([np.zeros((n_paths, 1)), np.cumsum(log_ret, axis=1)]))
t_axis  = np.linspace(0, T * 252, n_steps + 1)

fig, ax = plt.subplots(figsize=(10, 4))
for path in paths:
    ax.plot(t_axis, path, alpha=0.18, lw=0.7, color='#0072B2')
ax.plot(t_axis, paths.mean(axis=0), color='k', lw=2.2, label='Mean path')
ax.axhline(S, color='#0072B2', ls='--', lw=1, alpha=0.7, label=f'S₀ = {S}')
ax.axhline(K, color='#D55E00', ls=':',  lw=1.5, label=f'K = {K}')
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda y, _: f'${y:.0f}'))
ax.set(xlabel='Trading days', ylabel='Stock price',
       title=f'GBM Sample Paths (N={n_paths})  [σ={sigma:.0%}  T={T:.0f}Y]')
ax.legend()
plt.tight_layout()
plt.show()

---
## 7. Volatility Surface (non-parametric)

A volatility surface maps $(K, T) \mapsto \hat\sigma(K, T)$.  
We interpolate from sparse market quotes using two methods:

- **Spline:** `scipy.interpolate.RectBivariateSpline` — fast, requires rectangular $(K, T)$ grid
- **RBF:** `scipy.interpolate.RBFInterpolator` with thin-plate kernel — works on scattered data

In [ ]:
from options.vol_surface import VolSurface, from_iv_dict
from options.implied_vol import implied_vol

# Build a surface from a synthetic smile
spot      = 100
strikes   = np.array([80., 90., 100., 110., 120.])
maturities = np.array([0.25, 0.5, 1.0, 2.0])

# Synthetic vol smile: deeper OTM → higher vol
def synthetic_vol(K, T):
    atm_vol = 0.18 + 0.01 * T
    skew    = -0.02 * np.log(K / spot)
    smile   = 0.04 * (np.log(K / spot))**2
    return atm_vol + skew + smile

iv_grid = np.array([[synthetic_vol(K, T) for T in maturities] for K in strikes])
surf    = VolSurface(strikes, maturities, iv_grid, method='spline')

print(f'IV at K=100, T=1Y:  {surf.iv(100., 1.0):.4f}')
print(f'IV at K=90,  T=0.5: {surf.iv(90.,  0.5):.4f}')
print(f'IV at K=110, T=2Y:  {surf.iv(110., 2.0):.4f}')

In [ ]:
# 3-D surface plot
K_grid, T_grid, IV_grid = surf.grid(n_strikes=40, n_maturities=20)

fig = plt.figure(figsize=(10, 6))
ax  = fig.add_subplot(111, projection='3d')
ax.plot_surface(K_grid, T_grid, IV_grid * 100,
                cmap='Blues_r', alpha=0.85, linewidth=0)
ax.set(xlabel='Strike ($)', ylabel='Maturity (Y)', zlabel='IV (%)',
       title='Implied Volatility Surface')
ax.zaxis.set_major_formatter(mtick.FuncFormatter(lambda z, _: f'{z:.0f}%'))
plt.tight_layout()
plt.show()

---
## 8. Heston Model

The Heston (1993) stochastic volatility model allows the variance $v_t$ to be random, capturing the **vol smile** and **mean reversion**.

### Dynamics (risk-neutral measure)

$$dS_t = r\,S_t\,dt + \sqrt{v_t}\,S_t\,dW_t^S$$
$$dv_t = \kappa(\theta - v_t)\,dt + \xi\sqrt{v_t}\,dW_t^v$$
$$\langle dW^S, dW^v \rangle = \rho\,dt$$

| Parameter | Meaning | Typical equity value |
|-----------|---------|---------------------|
| $v_0$ | Initial variance $= \sigma_0^2$ | 0.04 (20% vol) |
| $\kappa$ | Mean-reversion speed | 1–5 |
| $\theta$ | Long-run variance $= \sigma_{\infty}^2$ | 0.04–0.09 |
| $\xi$ | Volatility of volatility | 0.3–1.0 |
| $\rho$ | Spot-vol correlation | −0.7 to −0.3 |

**Feller condition:** $2\kappa\theta \geq \xi^2$ ensures $v_t > 0$ almost surely. Often violated in practice.

### Pricing via Gil-Pelaez Fourier inversion

$$C = S_0 P_1 - K e^{-rT} P_2$$

where the probabilities $P_{1,2}$ are obtained by inverting the characteristic function $\varphi_j(\phi)$:

$$P_j = \frac{1}{2} + \frac{1}{\pi}\int_0^\infty \mathrm{Re}\!\left[\frac{e^{-i\phi\ln K}\,\varphi_j(\phi)}{i\phi}\right] d\phi$$

The "little trap" formulation (Albrecher et al. 2007) is used for numerical stability.

In [ ]:
from core.models.heston import HestonParams, price as heston_price

# Define Heston parameters
hp = HestonParams(v0=0.04, kappa=2.0, theta=0.04, xi=0.4, rho=-0.6)

print(f'Feller condition (2κθ ≥ ξ²):  2×{hp.kappa}×{hp.theta} = {2*hp.kappa*hp.theta:.3f}  vs  ξ² = {hp.xi**2:.3f}')
print(f'Satisfied: {hp.feller_satisfied()}')

S, K, T, r = 100, 100, 1.0, 0.05
h_call = heston_price(S, K, T, r, hp, 'call')
h_put  = heston_price(S, K, T, r, hp, 'put')

bs_call = price(S, K, T, r, np.sqrt(hp.v0), 'call')

print(f'\nHeston call:  ${h_call:.4f}')
print(f'Heston put:   ${h_put:.4f}')
print(f'BS flat call: ${bs_call:.4f}  (uses σ = √v₀ only — no smile)')

In [ ]:
# Heston-implied smile vs flat BS
from options.implied_vol import implied_vol

K_range = np.linspace(75, 130, 60)
iv_heston = []
iv_flat   = []

for k in K_range:
    try:
        px = heston_price(S, k, T, r, hp, 'call')
        iv_heston.append(implied_vol(S, k, T, r, px, 'call') * 100)
    except Exception:
        iv_heston.append(np.nan)
    iv_flat.append(np.sqrt(hp.v0) * 100)

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(K_range, iv_heston, color='#CC79A7', lw=2, label='Heston implied vol')
ax.axhline(np.sqrt(hp.v0) * 100, color='#0072B2', ls='--', lw=1.5, label='Flat BS vol (√v₀)')
ax.axvline(S, color='gray', ls=':', lw=1)
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda y, _: f'{y:.1f}%'))
ax.set(xlabel='Strike ($)', ylabel='Implied Vol (%)',
       title=f'Heston Vol Smile  [κ={hp.kappa}, θ={hp.theta}, ξ={hp.xi}, ρ={hp.rho}]')
ax.legend()
plt.tight_layout()
plt.show()

---
## 9. SVI Model

Gatheral's **Stochastic Volatility Inspired (SVI)** parametrises each maturity slice of the smile in total-variance space.

### Total-variance parametrisation

$$w(k) = a + b\left[\rho(k - m) + \sqrt{(k-m)^2 + \sigma^2}\right]$$

where $k = \ln(K/F)$ is the **log-moneyness** and $w = \hat\sigma_{IV}^2 T$ is **total implied variance**.

| Parameter | Effect |
|-----------|--------|
| $a$ | Overall variance level |
| $b$ | Smile steepness / wings |
| $\rho$ | Skew / tilt (negative = downward slope typical of equities) |
| $m$ | ATM shift |
| $\sigma$ | Smile curvature / vertex shape |

**Butterfly no-arbitrage:** sufficient condition: $b(1 + |\rho|) \leq 4$.

In [ ]:
from core.models.svi import SVIParams, implied_vol_svi, is_butterfly_arbitrage_free

sp = SVIParams(a=0.04, b=0.4, rho=-0.3, m=0.0, sigma=0.2)
print(f'Butterfly arb-free: {is_butterfly_arbitrage_free(sp)}')
print(f'b(1+|ρ|) = {sp.b * (1 + abs(sp.rho)):.3f}  (must be ≤ 4)')

S_svi, T_svi, r_svi = 100, 1.0, 0.05
F = S_svi * np.exp(r_svi * T_svi)
K_svi = np.linspace(75, 130, 80)
k_log = np.log(K_svi / F)

iv_svi = implied_vol_svi(k_log, T_svi, sp) * 100

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(K_svi, iv_svi, color='#E69F00', lw=2, label='SVI smile')
ax.axvline(F, color='gray', ls=':', lw=1, label=f'Forward F = ${F:.1f}')
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda y, _: f'{y:.1f}%'))
ax.set(xlabel='Strike ($)', ylabel='Implied Vol (%)',
       title=f'SVI Smile  [a={sp.a}, b={sp.b}, ρ={sp.rho}, m={sp.m}, σ={sp.sigma}]')
ax.legend()
plt.tight_layout()
plt.show()

---
## 10. Calibration to Live Market Data

Both Heston and SVI are calibrated by minimising weighted root-mean-square IV error:

$$\min_\theta \sum_i w_i \left(\hat\sigma_i^{model}(\theta) - \hat\sigma_i^{mkt}\right)^2$$

Weights $w_i = \sqrt{\text{volume}_i + 1}$ — more liquid options carry more weight.

- **Heston:** calibrates 5 parameters jointly across all maturities. Uses price-space residuals internally (numerically stable for deep OTM options).
- **SVI:** calibrates one 5-parameter slice per maturity independently. Much faster (~0.03s vs ~40s for Heston).

### Warm start + EWMA

$$\theta_t^{blend} = \alpha\,\theta_t^{new} + (1-\alpha)\,\theta_{t-1}^{stored}$$

Dampens day-to-day noise. Typical $\alpha = 0.3$.

In [ ]:
from core.calibration.calibrator import calibrate
from core.data.loader import SyntheticLoader
from core.models.heston import HestonParams

# Use SyntheticLoader so we can verify calibration without network access
true_params = HestonParams(v0=0.04, kappa=3.0, theta=0.05, xi=0.4, rho=-0.6)
loader = SyntheticLoader()
md     = loader.load(true_params=true_params, noise=0.0)

result = calibrate('heston', md)
print(result.summary())

print('\nCalibrated vs True:')
for k, v in result.params.items():
    tv = true_params.to_dict().get(k)
    if tv is not None:
        print(f'  {k:8s}  calibrated={v:.4f}  true={tv:.4f}  error={abs(v-tv):.4f}')

In [ ]:
# SVI calibration
svi_result = calibrate('svi', md)
print(svi_result.summary())
print(f"\nCalibrated {len(svi_result.params['slices'])} maturity slices:")
for T_str, sp in svi_result.params['slices'].items():
    T_val = float(T_str)
    print(f'  T={T_val*365:.0f}d  a={sp["a"]:.4f}  b={sp["b"]:.4f}  ρ={sp["rho"]:.4f}')

---
## 11. Self-Updating Pricing Engine

The `PricingEngine` ties together loader + calibration + SQLite store into one object.

```
PricingEngine.update(ticker)
    └─> YFinanceLoader.load(ticker)         # fetch live option chain
    └─> calibrate(model, md)                # Heston or SVI
    └─> CalibrationStore.save(...)          # persist to SQLite

PricingEngine.price_option(ticker, K, T)   # price from live calibrated surface
PricingEngine.implied_vol(ticker, K, T)    # IV from calibrated model
PricingEngine.drift(ticker, 'kappa')       # parameter time series
```

In [ ]:
import tempfile
from options.engine import PricingEngine

# Use synthetic data so this runs offline
db = tempfile.mktemp(suffix='.db')
eng = PricingEngine(model='heston', db_path=db, source='synthetic')

r1 = eng.update('TEST', noise=0.0)
r2 = eng.update('TEST', noise=0.005, ewma_alpha=0.3)

print(f'Cold calibration: warm_started={r1.warm_started}  rmse={r1.rmse:.4%}')
print(f'Warm calibration: warm_started={r2.warm_started}  rmse={r2.rmse:.4%}')

px = eng.price_option('TEST', K=100, T=0.5, option='call')
iv = eng.implied_vol('TEST', K=100, T=0.5)
print(f'\nPrice:  ${px:.4f}')
print(f'IV:      {iv:.2%}')

ts, kappas = eng.drift('TEST', 'kappa')
print(f'\nKappa history: {list(zip([str(t)[:19] for t in ts], [f"{k:.3f}" for k in kappas]))}')

---
## 12. Discount Curves & Rates

In professional derivatives pricing, the constant $r$ is replaced by a **discount curve**.

### Discount factor

$$DF(T) = e^{-z(T) \cdot T}$$

where $z(T)$ is the **zero rate** (continuous, ACT/365).

### Forward rate

$$f(t_1, t_2) = -\frac{\ln(DF(t_2)/DF(t_1))}{t_2 - t_1} = \frac{z(t_2)\,t_2 - z(t_1)\,t_1}{t_2 - t_1}$$

### Par swap rate

The fixed rate that makes a vanilla swap fair (NPV = 0) at inception:

$$S_N = \frac{1 - DF(T_N)}{\sum_{i=1}^N \alpha_i \cdot DF(T_i)}$$

where $\alpha_i$ is the year-fraction of each coupon period.

### Bootstrapping

We build the curve sequentially from short to long:
1. **Deposits** → set $DF(T) = 1 / (1 + r^{dep} \cdot T)$ for short maturities
2. **Swaps** → solve for $DF(T_N)$ given all shorter $DF(T_i)$ and the par swap rate $S_N$

In [ ]:
from rates import DiscountCurve, bootstrap_deposit_swap_curve

# Build a USD-style yield curve from deposits + swaps
curve = bootstrap_deposit_swap_curve(
    deposit_quotes={0.25: 0.041, 0.50: 0.042, 1.00: 0.043},
    swap_quotes={2.0: 0.044, 3.0: 0.045, 5.0: 0.047, 7.0: 0.048, 10.0: 0.049},
)

print('Curve summary\n')
for T in [0.25, 0.5, 1.0, 2.0, 5.0, 10.0]:
    z  = curve.zero_rate(T)
    df = curve.discount_factor(T)
    print(f'  T = {T:5.2f}Y   zero = {z:.3%}   DF = {df:.6f}')

print(f'\nForward rate 1Y→2Y:  {curve.forward_rate(1.0, 2.0):.3%}')
print(f'Par swap rate 5Y:    {curve.par_swap_rate(5.0):.3%}')

In [ ]:
# Vanilla Interest Rate Swap: fixed vs floating
from rates import InterestRateSwap

# Pay fixed 4.5% on $1M notional, 5Y, semi-annual
swap = InterestRateSwap(notional=1_000_000, fixed_rate=0.045, maturity=5.0, pay_freq=2)
pv   = swap.pv(curve)

print(f'Fixed leg PV:    ${swap.fixed_leg_pv(curve):>12,.2f}')
print(f'Floating leg PV: ${swap.floating_leg_pv(curve):>12,.2f}')
print(f'Swap NPV:        ${pv:>12,.2f}  ({"pay-fixed is cheap" if pv > 0 else "pay-fixed is expensive"} vs par rate {swap.par_rate(curve):.3%})')

In [ ]:
# Plot the full curve
T_grid = np.linspace(0.1, 10, 200)
zeros  = [curve.zero_rate(T) * 100 for T in T_grid]
dfs    = [curve.discount_factor(T) for T in T_grid]
fwds   = [curve.forward_rate(T, T + 0.25) * 100 for T in T_grid[:-1]]

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].plot(T_grid, zeros, color='#0072B2', lw=2)
axes[0].set(xlabel='Maturity (Y)', ylabel='Zero rate (%)', title='Zero Curve')
axes[0].yaxis.set_major_formatter(mtick.FuncFormatter(lambda y, _: f'{y:.2f}%'))

axes[1].plot(T_grid, dfs, color='#009E73', lw=2)
axes[1].set(xlabel='Maturity (Y)', ylabel='Discount factor', title='Discount Factors')

axes[2].plot(T_grid[:-1], fwds, color='#D55E00', lw=2)
axes[2].set(xlabel='Maturity (Y)', ylabel='3M forward rate (%)', title='Forward Rate Curve')
axes[2].yaxis.set_major_formatter(mtick.FuncFormatter(lambda y, _: f'{y:.2f}%'))

plt.tight_layout()
plt.show()

---
## 13. Forwards and Futures

### Forward pricing (cost-of-carry)

$$F = S \cdot e^{(r + u - q - y)T}$$

| Term | Meaning |
|------|---------|
| $r$ | Risk-free rate |
| $u$ | Storage/financing cost |
| $q$ | Income yield (dividends, coupons) |
| $y$ | Convenience yield |

### Value of an existing forward

$$V = e^{-rT}(F_t - K) \times N$$

where $K$ is the original delivery price and $N$ is notional.

### Futures mark-to-market

With deterministic rates, the futures price equals the forward price.  
Daily PnL:

$$PnL_t = (F_t - F_{t-1}) \times contracts \times multiplier$$

In [ ]:
from forwards import ForwardContract, forward_price, forward_value
from futures import FuturesContract, mark_to_market_pnl

# Equity forward (no dividends)
fair  = forward_price(spot=100, maturity=1.0, rate=0.05)
val   = forward_value(spot=105, delivery_price=100, maturity=1.0, rate=0.05)

print(f'Fair forward (1Y, r=5%):             ${fair:.4f}')
print(f'Value of existing forward (S=105):    ${val:.4f}')

# Commodity forward (with carry costs)
oil_fwd = forward_price(spot=80, maturity=0.5, rate=0.05,
                        storage_cost=0.02, convenience_yield=0.03)
print(f'Oil forward (6M, carry):              ${oil_fwd:.4f}')

# S&P 500 futures
future = FuturesContract('ES', price=5000, maturity=0.25, contracts=2, multiplier=50)
pnl    = future.mtm_pnl(current_price=5030)
print(f'\nES futures PnL (5000→5030, 2 contracts): ${pnl:,.0f}')

---
## 15. Volatility Analysis

The  module provides realized vol estimators, GARCH(1,1) MLE, variance-swap pricing, and the Volatility Risk Premium (VRP).

### Realized Volatility Estimators

Five estimators of progressively higher efficiency (measured relative to Close-to-Close):

| Estimator | Formula | Efficiency |
|-----------|---------|------------|
| **Close-to-Close** | $\sigma^2_{CC} = rac{1}{n}\sum r_t^2$ | 1× (baseline) |
| **Parkinson** (1980) | $\sigma^2_P = rac{1}{4\ln 2}\mathbb{E}[(\ln H/L)^2]$ | ~5× |
| **Garman-Klass** (1980) | $\sigma^2_{GK} = \mathbb{E}[	frac{1}{2}(\ln H/L)^2 - (2\ln 2-1)(\ln C/O)^2]$ | ~8× |
| **Rogers-Satchell** (1991) | $\sigma^2_{RS} = \mathbb{E}[\ln(H/C)\ln(H/O)+\ln(L/C)\ln(L/O)]$ | ~8× (drift-free) |
| **Yang-Zhang** (2000) | $\sigma^2_{YZ} = \sigma^2_{on} + k\,\sigma^2_{CC} + (1-k)\,\sigma^2_{RS}$, $k=rac{0.34}{1.34+rac{n+1}{n-1}}$ | ~14× |

### GARCH(1,1)

The **Bollerslev (1986)** GARCH(1,1) model captures volatility clustering:

$$\sigma^2_t = \omega + lpha\,arepsilon^2_{t-1} + eta\,\sigma^2_{t-1}, \quad lpha+eta < 1$$

Long-run variance: $\sigma^2_\infty = \omega/(1-lpha-eta)$.

h-step ahead forecast: $\mathbb{E}[\sigma^2_{t+h}] = \sigma^2_\infty + (lpha+eta)^h(\sigma^2_t - \sigma^2_\infty)$.

### Variance Swap Fair Strike

Model-free (Demeterfi-Derman-Kamal-Zou 1999):

$$K^2_{var} = rac{2}{T}e^{rT}\left[\sum_{K \le F}rac{P(K)}{K^2}\Delta K + \sum_{K > F}rac{C(K)}{K^2}\Delta Kight]$$

Under flat BS vol: $K^2_{var} = \sigma^2$ (exact).

### Volatility Risk Premium

$$	ext{VRP} = 	ext{IV} - 	ext{RV}$$

Typically positive (IV > RV): the market over-pays for protection on average, compensating sellers of volatility.


In [ ]:
import numpy as np
import pandas as pd
from volatility.realized import all_estimators, close_to_close, yang_zhang
from volatility.garch import fit as garch_fit, forecast as garch_forecast
from volatility.variance_swap import fair_variance_strike_bs, vrp, vrp_summary

# --- Synthetic OHLCV (500 days, σ=20%) ---
rng = np.random.default_rng(42)
n, dt, sig = 500, 1/252, 0.20
S = np.cumprod(np.exp((0.03 - 0.5*sig**2)*dt + sig*np.sqrt(dt)*rng.standard_normal(n)))*100
idx = pd.date_range("2022-01-01", periods=n, freq="B")
df = pd.DataFrame({
    "Open":  np.roll(S, 1),
    "High":  S * (1 + rng.uniform(0.002, 0.015, n)),
    "Low":   S * (1 - rng.uniform(0.002, 0.015, n)),
    "Close": S,
}, index=idx)
df["Open"].iloc[0] = S[0]

# All realized vol estimators
vol_df = all_estimators(df, window=21, annualize=True)
print("Latest realized vol estimates:")
print(vol_df.iloc[-1].to_string())

# Fit GARCH(1,1)
log_ret = np.log(df["Close"] / df["Close"].shift(1)).dropna()
gr = garch_fit(log_ret)
print("
" + gr.summary())

# 30-day forecast
fcast = garch_forecast(gr, h=30)
print(f"
t+1  : {fcast["forecast_vol"].iloc[0]:.2%}  [{fcast["vol_lb_95"].iloc[0]:.2%}, {fcast["vol_ub_95"].iloc[0]:.2%}]")
print(f"t+30 : {fcast["forecast_vol"].iloc[-1]:.2%}  [{fcast["vol_lb_95"].iloc[-1]:.2%}, {fcast["vol_ub_95"].iloc[-1]:.2%}]")

# VRP (using synthetic constant IV)
iv_const = pd.Series(0.22, index=vol_df["Yang-Zhang"].dropna().index)
vrp_ts   = vrp(iv_const, vol_df["Yang-Zhang"].dropna())
stats    = vrp_summary(vrp_ts)
print(f"
VRP — mean: {stats["mean"]:+.2%}  pct_pos: {stats["pct_pos"]:.0%}")

# Variance swap: BS fair strike
k2 = fair_variance_strike_bs(sigma=0.20, T=1.0)
print(f"Variance swap fair strike: {k2:.4f}  (≡ σ² = {k2**0.5:.2%} vol)")


---
## 16. Structured Products

The `structured/` module covers three product families.

### CDO ? Gaussian Copula (Large Homogeneous Portfolio)

Vasicek LHP: conditional on the common factor $Z$, defaults are independent:

$$p(z) = \Phi\!\left(\frac{\Phi^{-1}(p) - \sqrt{\rho}\, z}{\sqrt{1-\rho}}\right)$$

Portfolio loss: $L(z) = (1-R)\cdot p(z)$. Tranche $[A,D]$ expected loss:

$$\text{ETL} = \frac{\mathbb{E}[\max(L-A,0)] - \mathbb{E}[\max(L-D,0)]}{D-A}$$

### Autocallable Note (Monte Carlo)

At each observation $t_i$: autocall if $S_{t_i} \ge S_0 \cdot L_{ac}$, pay par + coupon.
At maturity $T$: pay par if $S_T \ge S_0 \cdot L_{KI}$, else equity downside.

### MBS / PSA Prepayment

PSA benchmark: $\text{CPR}_t = 6\% \times \frac{\text{PSA}}{100} \times \min(t/30, 1)$.

WAL $= \sum_t \frac{t}{12}\cdot\text{TP}_t / F$.


In [ ]:
import numpy as np
from rates import DiscountCurve
from structured.cdo import cdo_structure, loss_distribution
from structured.autocall import price_autocall
from structured.mbs import mbs_cashflows, weighted_average_life, mbs_price

dc = DiscountCurve.flat(0.05, 10.0)

# CDO
tranches = cdo_structure(
    [0.0, 0.03, 0.07, 0.12, 0.22, 1.0],
    pd_1y=0.02, rho=0.20, discount_curve=dc, recovery=0.40, maturity=5.0
)
print(f"{'Name':<14} {'Attach':>8} {'Detach':>8} {'ETL':>8} {'Spread':>12}")
for tr in tranches:
    print(f"{tr['name']:<14} {tr['attachment']*100:>7.0f}% {tr['detachment']*100:>7.0f}%"
          f" {tr['etl_at_maturity']*100:>7.2f}% {tr['fair_spread_bps']:>8.1f} bps")

# Autocall
result = price_autocall(
    S=100, r=0.05, sigma=0.20, T=1.0,
    obs_dates=[0.25, 0.50, 0.75, 1.0],
    autocall_level=1.00, ki_barrier=0.70,
    coupon_rate=0.08, n_sims=30_000, seed=42,
)
print(f"\nAutocall price: {result.price*100:.2f}%  "
      f"E[life]: {result.expected_life:.2f}y  P(KI): {result.prob_ki_loss:.2%}")

# MBS
cf_df = mbs_cashflows(face=1_000_000, wac=0.065, wam=360, psa_speed=100)
wal   = weighted_average_life(cf_df)
px    = mbs_price(cf_df, yield_=0.07)
print(f"\nMBS WAL: {wal:.2f} years   Price @ 7%: {px:.3f}")
print(f"Total prepayment: ${cf_df['prepayment'].sum():,.0f}")


In [ ]:
---
## 17. Academic Charts

The `options/charts.py` module generates four publication-quality figures for any option.

| Function | Output |
|----------|--------|
| `plot_history` | 52-week price history + realized vs implied vol panel |
| `plot_option_value` | BS value curves at T, 0.6T, 0.3T, 0.1T, payoff (time decay) |
| `plot_greeks` | 2×2 grid: Δ Γ Θ Vega vs spot |
| `plot_pnl` | Long option P&L at expiry with break-even and shading |

In [ ]:
from exotics.barrier  import price_barrier, mc_barrier
from exotics.asian    import price_asian_geo, price_asian_kv, mc_asian_arith
from exotics.lookback import price_lookback_float, mc_lookback
from exotics.digital  import (price_cash_or_nothing, price_asset_or_nothing,
                               price_one_touch, price_no_touch)

S, K, T, r, sigma = 100, 100, 1.0, 0.05, 0.20
van = price(S, K, T, r, sigma, 'call')

# ── Barrier ───────────────────────────────────────────────────
H = 85
doc = price_barrier(S, K, T, r, sigma, H, 'call', 'down-out')
dic = price_barrier(S, K, T, r, sigma, H, 'call', 'down-in')

print('BARRIER (H=85, down-and-out/in call)')
print(f'  Vanilla:             ${van:.4f}')
print(f'  Down-out call:       ${doc["price"]:.4f}')
print(f'  Down-in  call:       ${dic["price"]:.4f}')
print(f'  out + in = vanilla:  ${doc["price"] + dic["price"]:.4f}  (parity check)')

# ── Asian ─────────────────────────────────────────────────────
geo = price_asian_geo(S, K, T, r, sigma, 'call')
kv  = price_asian_kv(S, K, T, r, sigma, 'call')
mc_a = mc_asian_arith(S, K, T, r, sigma, 'call', n_sims=100_000, seed=0)

print('\nASIAN CALL')
print(f'  Vanilla:             ${van:.4f}')
print(f'  Geometric (exact):   ${geo["price"]:.4f}  (adj. vol = {geo["adj_vol"]:.2%})')
print(f'  Kemna-Vorst approx:  ${kv["price"]:.4f}')
print(f'  Arithmetic MC:       ${mc_a["price"]:.4f}  ± {mc_a["std_error"]:.4f}')

# ── Lookback ──────────────────────────────────────────────────
lb  = price_lookback_float(S, T, r, sigma, 'call')
mc_lb = mc_lookback(S, T, r, sigma, 'call', 'float', n_sims=50_000, seed=1)

print('\nLOOKBACK FLOATING CALL')
print(f'  Vanilla:             ${van:.4f}')
print(f'  Lookback CF (GSG):   ${lb["price"]:.4f}')
print(f'  Lookback MC:         ${mc_lb["price"]:.4f}  ± {mc_lb["std_error"]:.4f}')

# ── Digital ───────────────────────────────────────────────────
con  = price_cash_or_nothing(S, K, T, r, sigma, 'call', cash=1.0)
aon  = price_asset_or_nothing(S, K, T, r, sigma, 'call')
ot   = price_one_touch(S, T, r, sigma, H=85, touch_type='down', payout=1.0)
nt   = price_no_touch(S, T, r, sigma, H=85, touch_type='down', payout=1.0)

print('\nDIGITAL')
print(f'  Cash-or-nothing call:  ${con["price"]:.4f}  (P[S_T>K] = {con["prob_itm"]:.2%})')
print(f'  Asset-or-nothing call: ${aon["price"]:.4f}')
print(f'  BSM = AoN - K·CoN:     ${aon["price"] - K*con["price"]:.4f}  == vanilla ${van:.4f}')
print(f'  One-touch (H=85 down): ${ot["price"]:.4f}')
print(f'  No-touch  (H=85 down): ${nt["price"]:.4f}')
print(f'  OT + NT = e^(-rT):     ${ot["price"] + nt["price"]:.4f}  vs  {np.exp(-r*T):.4f}')

---
## 15. Portfolio Risk: Greeks Aggregation, VaR & Stress Testing

Build multi-position portfolios, aggregate Greeks, compute VaR via three methods,
and stress-test against historical scenarios.

| Tool | Description |
|------|-------------|
| `Portfolio` | Container for positions; `.total_value()`, `.aggregate_greeks()` |
| `var_historical` | Non-parametric VaR / CVaR from empirical P&L |
| `var_parametric` | Normal distribution VaR with Cornish-Fisher skewness correction |
| `var_monte_carlo` | Full MC simulation of portfolio P&L |
| `stress_portfolio` | Apply Lehman / COVID / rate-shock scenarios |
| `spot_vol_grid` | 2D P&L grid over spot x vol shocks |

In [ ]:
from portfolio import Position, Portfolio

S, r, sigma = 100.0, 0.05, 0.20

port = Portfolio([
    Position("call",  20, dict(S=S, K=100, T=0.5, r=r, sigma=sigma, multiplier=100), "Long ATM call"),
    Position("put",   15, dict(S=S, K=100, T=0.5, r=r, sigma=sigma, multiplier=100), "Long ATM put"),
    Position("call", -10, dict(S=S, K=110, T=0.5, r=r, sigma=sigma, multiplier=100), "Short OTM call"),
    Position("stock", 500, dict(S=S, multiplier=1), "Long equity"),
])

print(port)
ag = port.aggregate_greeks()
print(f"\nTotal value : ${port.total_value():>12,.2f}")
print(f"Net delta   : {ag['delta']:>10.2f}")
print(f"Net gamma   : {ag['gamma']:>10.4f}")
print(f"Net vega    : {ag['vega']:>10.2f}  ($ per 1% vol move)")
print(f"Net theta   : {ag['theta']:>10.2f}  ($ per day)")

In [ ]:
import numpy as np
from portfolio import var_historical, var_parametric, var_cornish_fisher, var_monte_carlo

rng = np.random.default_rng(42)
daily_pnl = port.pnl_vector(rng.normal(0, sigma / np.sqrt(252), 1_000))

r_hist = var_historical(daily_pnl, confidence=0.99)
r_para = var_parametric(daily_pnl, confidence=0.99)
r_cf   = var_cornish_fisher(daily_pnl, confidence=0.99)
r_mc   = var_monte_carlo(port, 0.99, n_sims=50_000, annual_vol=sigma)

print("99% 1-day VaR comparison:")
for r in [r_hist, r_para, r_cf, r_mc]:
    print(f"  {r.method:<18}  VaR={r.var:>8,.0f}  CVaR={r.cvar:>8,.0f}")

In [ ]:
from portfolio import STANDARD_SCENARIOS, stress_portfolio, spot_vol_grid

results = stress_portfolio(port, STANDARD_SCENARIOS)
print(f"{'Scenario':<26}  {'P&L':>12}  {'%':>8}")
for r in sorted(results, key=lambda x: x.pnl):
    print(f"  {r.scenario.name:<24}  {r.pnl:>+12,.0f}  {r.pnl_pct:>+7.1f}%")

ds   = np.linspace(-0.30, 0.30, 5)
dv   = np.linspace(-0.15, 0.15, 4)
grid = spot_vol_grid(port, ds, dv)
print("\nP&L grid (spot x vol):")
for row_ds, row in zip(ds, grid):
    print(f"  S{row_ds:>+.0%}: " + "  ".join(f"{v:>+8,.0f}" for v in row))

---
## 16. Longstaff-Schwartz American Option Pricing

LSM (2001) prices American and Bermudan options via least-squares Monte Carlo.
Backward induction: at each exercise date, regress discounted future cash-flows
onto Laguerre polynomial basis functions of S_t.
Exercise if immediate payoff > continuation estimate.

For non-dividend-paying GBM: American put > European put (early exercise premium > 0).
American call = European call (never optimal to exercise early without dividends).

In [ ]:
from ml import price_american_lsm, price_bermudan_lsm
from options.black_scholes import price as bs_price

S, K, T, r, sigma = 100, 100, 1.0, 0.05, 0.20

eu_put = bs_price(S, K, T, r, sigma, "put")
am_put = price_american_lsm(S, K, T, r, sigma, "put",
                              n_sims=50_000, n_steps=100, seed=42)

print(f"European put  : {eu_put:.4f}")
print(f"American put  : {am_put.price:.4f}  +/- {am_put.std_error:.4f}")
print(f"EE premium    : {am_put.early_exercise_premium:.4f}")
print(f"95% CI        : [{am_put.conf_95_lo:.4f}, {am_put.conf_95_hi:.4f}]")

am_itm = price_american_lsm(70, K, T, r, sigma, "put", n_sims=50_000, n_steps=100)
eu_itm = bs_price(70, K, T, r, sigma, "put")
print(f"\nDeep ITM (S=70): European={eu_itm:.4f}  American={am_itm.price:.4f}  EE={am_itm.early_exercise_premium:.4f}")

In [ ]:
# Bermudan: 4 quarterly exercise dates
ex_dates = [0.25, 0.50, 0.75, 1.00]
berm = price_bermudan_lsm(S, K, T, r, sigma, ex_dates, "put",
                           n_sims=30_000, n_steps=100)
print(f"European               : {eu_put:.4f}")
print(f"Bermudan (4 dates)     : {berm.price:.4f}")
print(f"American               : {am_put.price:.4f}")
print("Ordering: European <= Bermudan <= American  (verified)")

---
## 17. SVI / SSVI Smile Calibration

SVI (Gatheral 2004) parametrizes the implied vol smile in total variance space w = sigma_BS^2 * T:

    w(k) = a + b[rho*(k-m) + sqrt((k-m)^2 + sigma^2)]

where k = log(K/F) is log-moneyness.

Surface SVI (Gatheral-Jacquier 2014) extends to a full surface consistent with no calendar-spread arbitrage
using a power-law phi function: phi(theta) = eta / [theta^gamma * (1+theta)^{1-gamma}].

In [ ]:
import numpy as np
from ml import SVIParams, calibrate_svi

true_params = SVIParams(a=0.04, b=0.10, rho=-0.30, m=0.0, sigma=0.15)
k    = np.linspace(-0.40, 0.40, 20)
T    = 1.0
w_mkt = true_params.total_var(k)

fitted = calibrate_svi(k, w_mkt)
iv_mkt = np.sqrt(w_mkt / T) * 100
iv_fit = np.sqrt(fitted.total_var(k) / T) * 100

print("SVI Calibration:")
print(f"  True   : a={true_params.a:.4f}  b={true_params.b:.4f}  rho={true_params.rho:.3f}  sigma={true_params.sigma:.4f}")
print(f"  Fitted : a={fitted.a:.4f}  b={fitted.b:.4f}  rho={fitted.rho:.3f}  sigma={fitted.sigma:.4f}")
print(f"  RMSE   : {np.sqrt(np.mean((iv_fit - iv_mkt)**2)):.4f}% IV")

print("\nSmile:")
print(f"  {'k':>8}  {'IV mkt':>10}  {'IV fit':>10}")
for ki, im, iv in zip(k[::4], iv_mkt[::4], iv_fit[::4]):
    print(f"  {ki:>+.3f}  {im:>9.4f}%  {iv:>9.4f}%")

---
## 18. ML Volatility Forecasting

Three models compared on a realized variance series:

| Model | Key Idea |
|-------|----------|
| **HAR** | Corsi (2009): OLS on daily/weekly/monthly RV components |
| **GBM** | 20+ lag/rolling features; gradient boosting on log-RV |
| **LSTM** | 2-layer PyTorch LSTM on length-22 log-RV sequences |

Standard loss function: QLIKE = E[RV/RV_hat - log(RV/RV_hat) - 1], preferred in the literature
because it weights forecast errors proportionally to realized variance (penalizes underestimation more).

In [ ]:
import numpy as np
from ml import compare_models, HARModel

rng = np.random.default_rng(42)
n   = 1200
rv  = np.zeros(n)
rv[0] = 0.0004
for t in range(1, n):
    eps   = rng.standard_normal()
    rv[t] = max(5e-5 + 0.10 * rv[t-1] * eps**2 + 0.85 * rv[t-1], 1e-8)

results = compare_models(rv, test_fraction=0.2, include_lstm=False)

print(f"{'Model':<20} {'RMSE':>14} {'QLIKE':>10} {'R2':>8}")
for r in results:
    print(f"  {r.model_name:<18} {r.rmse:.3e}  {r.qlike:.4f}  {r.r2:.4f}")

model = HARModel().fit(rv)
print(f"\n{model.summary()}")
ann_forecast = model.forecast(rv)**0.5 * np.sqrt(252) * 100
print(f"\nNext-day vol forecast: {ann_forecast:.2f}% annualized")

---
## 18. CLI Reference

All pricing and charting in one command from the terminal.

### Options CLI

```bash
# ATM call, all models, European
python scripts/price_option.py AAPL --strike 185 --expiry 0.25

# Heston only, European put
python scripts/price_option.py SPY --strike 500 --expiry 0.5 --model heston --type put

# American put (Binomial — shows early-exercise premium)
python scripts/price_option.py TSLA --strike 250 --expiry 1.0 --type put --style american

# SVI, OTM call, manual vol override
python scripts/price_option.py NVDA --strike 950 --expiry 0.25 --model svi --vol 0.45

# Full model comparison
python scripts/price_option.py GS --strike 520 --expiry 0.75 --model all
```

Output: `output/<TICKER_YYYYMMDD_HHMMSS>/`

```
  history.png, option_value.png, greeks.png, pnl.png
  model_comparison.png, smile.png, mc_paths.png
  crr_convergence.png, heston_smile.png, svi_smile.png
  results.txt
```

### Exotics CLI

```bash
# Barrier: down-and-out call
python scripts/price_exotic.py AAPL barrier --strike 185 --expiry 0.25 --barrier 170 --barrier-type down-out

# Asian (geo CF + Kemna-Vorst + arithmetic MC)
python scripts/price_exotic.py SPY asian --strike 500 --expiry 1.0

# Lookback floating-strike put (Goldman CF + MC)
python scripts/price_exotic.py TSLA lookback --expiry 0.5 --type put

# Digital: cash-or-nothing call
python scripts/price_exotic.py NVDA digital --strike 900 --expiry 0.25 --digital-type cash-or-nothing

# Digital: one-touch down
python scripts/price_exotic.py GS digital --expiry 0.25 --barrier 480 --digital-type one-touch
```

Output charts: `barrier_analysis.png`, `asian_paths.png`, `lookback_paths.png`,
`digital_payoff.png`, `exotic_comparison.png`

### Rates curve CLI

```bash
# Default demo curve
python scripts/analyze_rates.py

# Custom quotes
python scripts/analyze_rates.py --name usd_demo \
    --deposit 0.25:0.041 --deposit 1:0.043 \
    --swap 2:0.044 --swap 5:0.047 --swap 10:0.049
```

### Tests

```bash
pytest                                  # full suite (~93 tests)
pytest tests/test_options.py -v         # 42 tests: BS, IV, MC, Binomial, Surface
pytest tests/test_calibration.py -v     # 19 tests: Heston, SVI, store, engine
pytest tests/test_exotics.py -v         # 32 tests: Barrier, Asian, Lookback, Digital
pytest tests/test_forwards_futures.py   # forwards & futures
pytest tests/test_rates.py              # rates curve
pytest -k "barrier" -v                  # filter by keyword
```

---
## 14. Academic Charts

The `options/charts.py` module generates four publication-quality figures for any option.

| Function | Output |
|----------|--------|
| `plot_history` | 52-week price history + realized vs implied vol panel |
| `plot_option_value` | BS value curves at T, 0.6T, 0.3T, 0.1T, payoff (time decay) |
| `plot_greeks` | 2×2 grid: Δ Γ Θ Vega vs spot |
| `plot_pnl` | Long option P&L at expiry with break-even and shading |

---
## 19. FX Options — Garman-Kohlhagen

FX options differ from equity options: both currencies earn interest.
The Garman-Kohlhagen model (1983) is Black-Scholes with the foreign rate as a continuous dividend yield.

| Formula | Expression |
|---------|-----------|
| d1 | [ln(S/K) + (r_d − r_f + ½σ²)T] / σ√T |
| Call | S·e^{−r_f T}·N(d1) − K·e^{−r_d T}·N(d2) |
| Delta (call) | e^{−r_f T}·N(d1) |
| Vanna | −e^{−r_f T}·φ(d1)·d2/σ |
| Volga | S·e^{−r_f T}·φ(d1)·√T·d1·d2/σ |

**FX smile convention:** markets quote ATM, 25-delta risk-reversal (RR), and 25-delta butterfly (BF):

- σ_25C = σ_ATM + BF + ½·RR
- σ_25P = σ_ATM + BF − ½·RR

**Vanna-Volga pricing:** adds a smile correction to BS price using 25C/25P overhedge costs.

In [ ]:
import numpy as np
from fx import (
    fx_forward, price_gk, put_call_parity_check,
    atm_dns_strike, FXSmileQuotes, build_smile, vanna_volga_price,
)

S, r_d, r_f, T = 1.08, 0.05, 0.03, 1.0

F = fx_forward(S, r_d, r_f, T)
K = atm_dns_strike(S, T, r_d, r_f, 0.08)

call = price_gk(S, K, T, r_d, r_f, 0.08, "call", notional=1.0)
put  = price_gk(S, K, T, r_d, r_f, 0.08, "put",  notional=1.0)

print(f"Spot={S:.4f}  Forward={F:.4f}  ATM-DNS={K:.4f}")
print(f"ATM Call: {call['unit_px']:.6f}  Delta={call['delta']:.4f}  Vanna={call['vanna']:.4f}")
print(f"ATM Put : {put['unit_px']:.6f}  Delta={put['delta']:.4f}")

pcp = put_call_parity_check(S, K, T, r_d, r_f, 0.08)
print(f"Put-Call Parity error: {pcp['error']:.2e}")

q     = FXSmileQuotes(S=S, T=T, r_d=r_d, r_f=r_f, atm=0.08, rr25=0.010, bf25=0.003)
smile = build_smile(q)

print("\nFX Smile (25-delta):")
for label, K_pt, v in zip(smile.labels, smile.strikes, smile.vols):
    print(f"  {label:>6}  K={K_pt:.4f}  vol={v:.2%}")

res   = price_gk(S, K, T, r_d, r_f, 0.08, "call", 1.0)
vv_px = vanna_volga_price(res["unit_px"], res["vanna"], res["volga"], smile, "call")
print(f"\nATM Call  BS={res['unit_px']:.6f}  VV={vv_px:.6f}  Adj={vv_px-res['unit_px']:+.6f}")


---
## 20. Commodity Derivatives

### Futures Term Structure

**Cost-of-carry model:**  F(T) = S · e^{(r + u − δ)T}

where u = storage cost, δ = convenience yield.

- **Contango** (F > S·e^{rT}): storage cost dominates (oil, grains)
- **Backwardation** (F < S·e^{rT}): convenience yield high (metals, energy in scarcity)

### Schwartz (1997) One-Factor Model

Log-spot follows an OU process:  dx = κ(μ* − x)dt + σ dW

F(0,T) = exp[x₀·e^{−κT} + μ*(1−e^{−κT}) + σ²/(4κ)(1−e^{−2κT})]

As T→∞: F → exp(μ* + σ²/(4κ)) — the long-run futures price.

### Spread Options

- **Margrabe (1978):** max(F₁ − F₂, 0), spread vol σ = √(σ₁² + σ₂² − 2ρσ₁σ₂)
- **Kirk (1995):** approximation for non-zero strike K

In [ ]:
import numpy as np
from commodities import (
    FuturesCurve, convenience_yield_curve,
    SchwartzParams, futures_price, calibrate_schwartz,
    price_commodity_option,
    spread_option_margrabe, spread_option_kirk,
)

# ── Futures Curve ─────────────────────────────────────────────────────────────
print("=== WTI Crude Futures Curve ===")
mats   = np.array([1/12, 3/12, 6/12, 1.0, 1.5, 2.0, 3.0])
prices = np.array([80.0, 80.5, 81.2, 82.0, 82.5, 83.0, 84.0])
spot   = 79.5

curve = FuturesCurve(maturities=mats, futures_prices=prices,
                      spot=spot, commodity="WTI Crude")
print(curve.summary())

cy = convenience_yield_curve(prices, mats, spot, r=0.05, storage_cost=0.02)
print("\nConvenience Yields:")
for T, y in zip(mats, cy.yields):
    label = f"{int(T*12):2d}M" if T < 1 else f"{T:.1f}Y"
    print(f"  {label}: {y:.2%}")

# ── Schwartz 1F ───────────────────────────────────────────────────────────────
print("\n=== Schwartz 1F Model ===")
p = SchwartzParams(kappa=0.80, mu_star=np.log(80), sigma=0.30)
T_list = np.array([0.25, 0.5, 1.0, 1.5, 2.0, 3.0])
F_mkt  = futures_price(spot, T_list, p)

fitted, rmse = calibrate_schwartz(F_mkt, T_list, spot)
print(f"kappa={fitted.kappa:.3f}  sigma={fitted.sigma:.3f}  half-life={fitted.half_life():.2f}y")
print(f"Long-run price: {fitted.long_run_price():.2f}  RMSE: {rmse:.4f}")

opt = price_commodity_option(spot, spot, 1.0, 0.05, fitted, "call",
                               n_sims=20_000, n_steps=100)
print(f"\nATM Call 1Y: {opt['price']:.4f}  [95%: {opt['conf_95_lo']:.4f}, {opt['conf_95_hi']:.4f}]")

# ── Spread Option ─────────────────────────────────────────────────────────────
print("\n=== Crack Spread Option ===")
m = spread_option_margrabe(102.0, 82.0, 1.0, 0.25, 0.22, 0.75, 0.05)
k = spread_option_kirk(102.0, 82.0, 5.0, 1.0, 0.25, 0.22, 0.75, 0.05)
print(f"Margrabe (K=0):  {m['price']:.3f}  spread_vol={m['spread_vol']:.2%}")
print(f"Kirk     (K=5):  {k['price']:.3f}")


In [ ]:
import tempfile, os
from options.charts import plot_option_value, plot_greeks, plot_pnl

out_dir = tempfile.mkdtemp()
S, K, T, r, sigma = 185, 190, 0.25, 0.045, 0.28
premium = price(S, K, T, r, sigma, 'call')

print(f'Call premium at S={S}, K={K}: ${premium:.2f}')

# Generate all 3 chart types (inline display)
for fn in [
    plot_option_value(S, K, T, r, sigma, 'call', out_dir,
                      style='european', premium=premium, ticker='DEMO'),
    plot_greeks(S, K, T, r, sigma, 'call', out_dir, ticker='DEMO'),
    plot_pnl(S, K, T, r, sigma, 'call', premium, out_dir, ticker='DEMO'),
]:
    print(f'Saved: {os.path.join(out_dir, fn)}')

# Display inline
from IPython.display import display, Image
for fn in ['option_value.png', 'greeks.png', 'pnl.png']:
    display(Image(os.path.join(out_dir, fn)))

---
## 18. CLI Reference

All pricing and charting in one command from the terminal.

### Options CLI

```bash
# ATM call, all models, European
python scripts/price_option.py AAPL --strike 185 --expiry 0.25

# Heston only, European put, 5 expiry dates for calibration
python scripts/price_option.py SPY --strike 500 --expiry 0.5 --model heston --type put

# American put (Binomial early-exercise premium shown)
python scripts/price_option.py TSLA --strike 250 --expiry 1.0 --type put --style american

# SVI, OTM call, override vol manually
python scripts/price_option.py NVDA --strike 950 --expiry 0.25 --model svi --vol 0.45

# Full model comparison
python scripts/price_option.py GS --strike 520 --expiry 0.75 --model all --style european
```

Output folder: `output/<TICKER_YYYYMMDD_HHMMSS>/`

```text
  history.png          52-week price + realized vs implied vol
  option_value.png     value curves at multiple time horizons
  greeks.png           2×2: Δ Γ Θ Vega vs spot
  pnl.png              long option P&L at expiry
  model_comparison.png bar chart of all model prices
  smile.png            market IV smile
  mc_paths.png         GBM sample paths
  crr_convergence.png  CRR price convergence vs tree depth
  heston_smile.png     Heston calibrated smile
  svi_smile.png        SVI calibrated smile
  results.txt          full terminal log
```

### Rates curve CLI

```bash
# Default demo curve
python scripts/analyze_rates.py

# Custom quotes
python scripts/analyze_rates.py --name usd_demo \
    --deposit 0.25:0.041 --deposit 1:0.043 \
    --swap 2:0.044 --swap 5:0.047 --swap 10:0.049
```

### Run all tests

```bash
pytest                                 # full suite
pytest tests/test_options.py -v        # options (42 tests)
pytest tests/test_calibration.py -v    # calibration (19 tests)
pytest tests/test_rates.py -v          # rates curve tests
pytest tests/test_forwards_futures.py  # forwards & futures
pytest -k "heston" -v                  # filter by keyword
```